When acquiring bonds, the accrued interest must be calculated. While the
logic for calculating this interest is fairly straightforward,

$$
\text{accrued interest} = \text{amount} \cdot i \cdot \text{year_fraction}
$$

determining the day count as a fraction of the year must take into account
various day-count conventions. This makes the task perfectly
suited to a Python function. 

While we could write all the necessary code ourselves, two Python
libraries already exist to handle dates according to our needs:
`datetime` and `QuantLib`. The remaining challenge lies in bridging the
gap between the different date formats required by the two libraries.
Specifically, `datetime` expects dates in the format `YYYY, M, D`,
whereas `QuantLib` requires `D, M, YYYY`. 

`datetime` is part of Python's standard library, whereas `QuantLib` must
be installed separately via a package manager (e.g. pip).
Both libraries need to be imported to be used in the following code.

In [1]:
from datetime import date
import QuantLib as ql

We could pass `QuantLib` dates as inputs to the Python
function. However, to ensure compatibility with the standard Python
date format, it is preferable to use the `datetime.date` data type.
Therefore we need auxiliary functions to bridge the two formatting
conventions. 

In [ ]:
def to_ql(d: date) -> ql.Date:
    """Converts a Python date into a QuantLib date.

    Pay attention to the argument order: Python uses
    date(year, month, day), whereas QuantLib uses ql.Date(day, month, year).

    Args:
        d: A Python datetime.date object.

    Returns:
        The corresponding QuantLib ql.Date object.
    """
    return ql.Date(d.day, d.month, d.year)


def to_py(d: ql.Date) -> date:
    """Converts a QuantLib date into a Python date.

    Args:
        d: A QuantLib ql.Date object.

    Returns:
        The corresponding Python datetime.date object.
    """
    return date(d.year(), d.month(), d.dayOfMonth())

The `QuantLib` library offers multiple counting conventions. The most
widely used convention in Switzerland is the "30E/360 German" counting
convention. Hence, we set this as default to our Python
`get_accrued_interest()` function.

In [ ]:
def get_accrued_interest(interest_rate: float, amount: float,
                         start_date: date, end_date: date,
                         percent: bool = False,
                         day_counter: ql.DayCounter = None) -> float:
    """Calculates the accrued interest of a bond.

    Args:
        interest_rate: Coupon rate, as a decimal (0.04) or percentage (4).
        amount: Nominal amount (face value) of the bond.
        start_date: Last coupon date.
        end_date: Value date (settlement date).
        percent: True if interest_rate is provided as a percentage.
        day_counter: Day-count convention; default is the German convention.

    Returns:
        The accrued interest.

    Raises:
        ValueError: If end_date is before start_date.
    """
    if end_date < start_date:
        raise ValueError("end_date must be after start_date.")
    if day_counter is None:
        day_counter = ql.Thirty360(ql.Thirty360.German)

    i = interest_rate / 100 if percent else interest_rate
    year_fraction = day_counter.yearFraction(to_ql(start_date), to_ql(end_date))
    return amount * i * year_fraction

Other counting conventions are listed in the table below.

| Convention | Code |
| :--- | :--- |
| 30E/360 German | ql.Thirty360(ql.Thirty360.German) |
| 30E/360 Euro | ql.Thirty360(ql.Thirty360.European) |
| Act/360 | ql.Actual360() |
| Act/365 Fixed | ql.Actual365Fixed() |
| Act/Act ISDA | ql.ActualActual(ql.ActualActual.ISDA) |